# 05 — Fama-French SMB & HML Replication from JKP Data

Replicates the Fama-French 2×3 SMB and HML factors using only the JKP
global factor dataset, following the exact FF methodology:

1. **Universe**: US common stocks, primary securities, CRSP-sourced,
   positive ME and positive BE/ME, at least 2 years of history.
2. **Annual June rebalance**: sort stocks in June each year, hold
   portfolios from July of year *t* through June of year *t+1*.
3. **NYSE-only breakpoints**: size median, BE/ME 30th/70th percentiles.
4. **Value-weighted** returns using `market_equity` as weight.
5. **Return**: `ret_exc_lead1m` (next-month excess return from JKP).

Then compares against the official FF factors: correlation, regression
beta/R², and cointegration.

### JKP vs CRSP/Compustat universe mapping

| FF (your CRSP code) | JKP equivalent | Notes |
|---------------------|----------------|-------|
| `shrcd in (10,11)` | `common=1` + `primary_sec=1` | JKP pre-filters to common shares |
| `exchcd between 1 and 3` | `exch_main in (1,2,3)` | 1=NYSE, 2=AMEX, 3=NASDAQ |
| `exchcd == 1` (for breakpoints) | `exch_main == 1` | NYSE only |
| `count >= 1` (2+ yrs Compustat) | Approximated via `gvkey` history | |
| `me > 0` and `beme > 0` | `market_equity > 0` and `be_me > 0` | |
| Book equity from Compustat | `be_me` pre-computed by JKP | |
| `retadj` (with delisting) | `ret_exc_lead1m` | JKP incorporates delisting returns |

**Timing note**: `ret_exc_lead1m` at `eom=June` is the return realized in
July. After computing VW portfolio returns, we shift dates forward by 1 month
so the factor return is dated to the month it was realized — matching the FF
convention.

In [ ]:
# ── Environment setup (run once) ──────────────────────────────────────
import importlib, sys, os

if "google.colab" in sys.modules:
    from google.colab import drive
    drive.mount("/content/drive")
    if not os.path.exists("/content/quant-trading-experiments"):
        !git clone https://github.com/gunsslashroses/quant-trading-experiments.git /content/quant-trading-experiments
    %cd /content/quant-trading-experiments
    !pip install -q -e ".[dev]"
    JKP_CSV_PATH = "/content/drive/MyDrive/jkp_data.csv"
else:
    JKP_CSV_PATH = "/Users/abhin/Library/CloudStorage/GoogleDrive-abhinavkeshri1999@gmail.com/My Drive/Colab Notebooks/CPSC 381-581: Machine Learning/Final project/jkp_data.csv"

print(f"Data path: {JKP_CSV_PATH}")

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

from quant_trading.data import load_jkp_csv

## 1. Load and filter JKP data to FF universe

In [ ]:
# Load raw JKP data
df = load_jkp_csv(JKP_CSV_PATH)
print(f"Raw JKP rows: {len(df):,}")
print(f"Date range: {df['month_date'].min()} to {df['month_date'].max()}")
print(f"Columns: {list(df.columns)}")

In [ ]:
# ── FF universe filters ───────────────────────────────────────────────
# Mirror the CRSP/Compustat filters from the reference code:
#   shrcd in (10,11) → common=1 (already filtered in JKP SQL) + primary_sec=1
#   exchcd in (1,2,3) → exch_main in (1,2,3)
#   me > 0 and beme > 0
#   count >= 1 (at least 2 years of Compustat history)

# 1. Primary securities on major exchanges, CRSP-sourced
df_ff = df[
    (df["primary_sec"] == 1) &
    (df["exch_main"].isin([1, 2, 3])) &
    (df["source_crsp"] == 1)
].copy()
print(f"After primary_sec + exchange + CRSP filter: {len(df_ff):,}")

# 2. Require positive market equity and positive book-to-market
df_ff = df_ff[
    (df_ff["market_equity"] > 0) &
    (df_ff["be_me"] > 0)
].copy()
print(f"After ME>0 and BE/ME>0: {len(df_ff):,}")

# 3. Approximate "at least 2 years in Compustat" by counting how many
#    months each gvkey has appeared before the current month.
#    FF requires count>=1 (0-indexed, so 2+ fiscal years).
#    We approximate: if a stock has appeared for 24+ months, it qualifies.
df_ff = df_ff.sort_values(["id", "month_date"])
df_ff["obs_count"] = df_ff.groupby("id").cumcount()
df_ff = df_ff[df_ff["obs_count"] >= 24].copy()
print(f"After 2-year history filter: {len(df_ff):,}")

# 4. Extract year and month for the annual rebalance logic
df_ff["year"] = df_ff["month_date"].dt.year
df_ff["month"] = df_ff["month_date"].dt.month

print(f"\nFinal FF universe: {len(df_ff):,} stock-months")
print(f"Unique stocks: {df_ff['id'].nunique():,}")
print(f"Date range: {df_ff['month_date'].min()} to {df_ff['month_date'].max()}")

## 2. Annual June sort with NYSE breakpoints

FF methodology:
- Each June, sort all eligible stocks using **NYSE-only** breakpoints.
- Size: NYSE median of `market_equity`.
- Value: NYSE 30th and 70th percentiles of `be_me`.
- Hold portfolios from **July of year t** through **June of year t+1**.

In [ ]:
# ── Step 2a: Compute NYSE breakpoints each June ──────────────────────
june_data = df_ff[df_ff["month"] == 6].copy()
nyse_june = june_data[june_data["exch_main"] == 1].copy()

# Size breakpoint: NYSE median market equity
size_breaks = (
    nyse_june.groupby("year")["market_equity"]
    .median()
    .rename("size_median")
    .reset_index()
)

# Value breakpoints: NYSE 30th and 70th percentiles of BE/ME
bm_breaks = (
    nyse_june.groupby("year")["be_me"]
    .quantile([0.3, 0.7])
    .unstack()
    .rename(columns={0.3: "bm30", 0.7: "bm70"})
    .reset_index()
)

breaks = pd.merge(size_breaks, bm_breaks, on="year")
print(f"Breakpoints computed for {len(breaks)} years")
breaks.head()

In [ ]:
# ── Step 2b: Assign portfolios in June ────────────────────────────────
june_sorted = pd.merge(june_data, breaks, on="year", how="left")

# Size bucket: S or B
june_sorted["sz"] = np.where(
    june_sorted["market_equity"] <= june_sorted["size_median"], "S", "B"
)

# BE/ME bucket: L, M, or H
june_sorted["bm"] = np.where(
    june_sorted["be_me"] <= june_sorted["bm30"],
    "L",
    np.where(june_sorted["be_me"] <= june_sorted["bm70"], "M", "H"),
)

june_sorted["portfolio"] = june_sorted["sz"] + june_sorted["bm"]

# Keep only the assignment columns
june_assignments = june_sorted[["id", "year", "sz", "bm", "portfolio"]].copy()

# This assignment applies from July of year t to June of year t+1.
# We create an "ffyear" column: for July-Dec it's the current year,
# for Jan-June it's the previous year.
june_assignments = june_assignments.rename(columns={"year": "sort_year"})

print(f"June assignments: {len(june_assignments):,} stock-years")
print(f"\nPortfolio counts (pooled across years):")
print(june_assignments["portfolio"].value_counts().sort_index())

In [ ]:
# ── Step 2c: Map assignments to monthly returns (July t → June t+1) ──

# Create the FF fiscal year: July-Dec = current year, Jan-June = prior year
df_ff["ffyear"] = np.where(
    df_ff["month"] >= 7,
    df_ff["year"],
    df_ff["year"] - 1,
)

# Merge: each stock-month gets the portfolio assigned in June of its ffyear
df_monthly = pd.merge(
    df_ff,
    june_assignments,
    left_on=["id", "ffyear"],
    right_on=["id", "sort_year"],
    how="inner",
)

# Drop the sort month itself (June) — FF holds July through next June
# Actually, FF portfolios include July of year t through June of year t+1
# The return in June of year t is NOT part of the portfolio.
# month>=7 in ffyear=t OR month<=6 in ffyear=t (which is year t+1 months 1-6)
# The merge already handles this correctly via ffyear.

print(f"Monthly returns with portfolio assignments: {len(df_monthly):,}")
print(f"Unique stocks: {df_monthly['id'].nunique():,}")
print(f"Portfolios: {sorted(df_monthly['portfolio'].unique())}")

## 3. Compute value-weighted portfolio returns

In [ ]:
# ── Value-weighted returns per portfolio per month ─────────────────────
#
# IMPORTANT TIMING NOTE:
# ret_exc_lead1m at month_date=t is the return realized from t to t+1.
# So at month_date=June, ret_exc_lead1m = the July return.
# At month_date=July, ret_exc_lead1m = the August return. Etc.
#
# Weight: market_equity at month_date=t is end-of-month-t market cap,
# which is beginning-of-period for the return from t to t+1. Correct.
#
# After computing VW returns grouped by month_date, we shift the index
# forward by 1 month so that the factor return is dated to the month
# when it was actually realized. This aligns with FF's dating convention.

def vw_return(g):
    """Value-weighted return for a group."""
    valid = g.dropna(subset=["ret_exc_lead1m", "market_equity"])
    if valid.empty or valid["market_equity"].sum() == 0:
        return np.nan
    return (
        (valid["ret_exc_lead1m"] * valid["market_equity"]).sum()
        / valid["market_equity"].sum()
    )


port_rets = (
    df_monthly.groupby(["month_date", "portfolio"])
    .apply(vw_return, include_groups=False)
    .unstack("portfolio")
)

# Shift index forward 1 month: the return computed from month_date=t data
# is actually realized in month t+1.
port_rets.index = port_rets.index + pd.DateOffset(months=1)
port_rets.index.name = "date"

# Verify we have all 6 portfolios
expected = ["BH", "BL", "BM", "SH", "SL", "SM"]
assert all(p in port_rets.columns for p in expected), \
    f"Missing portfolios: {set(expected) - set(port_rets.columns)}"

print(f"Portfolio returns: {len(port_rets)} months")
print(f"Date range: {port_rets.index.min()} to {port_rets.index.max()}")
port_rets.head()

In [ ]:
# ── Construct SMB and HML ─────────────────────────────────────────────
# SMB = (SH + SM + SL)/3 − (BH + BM + BL)/3
# HML = (SH + BH)/2 − (SL + BL)/2

factors = pd.DataFrame(index=port_rets.index)
factors["SMB"] = (
    (port_rets["SH"] + port_rets["SM"] + port_rets["SL"]) / 3
    - (port_rets["BH"] + port_rets["BM"] + port_rets["BL"]) / 3
)
factors["HML"] = (
    (port_rets["SH"] + port_rets["BH"]) / 2
    - (port_rets["SL"] + port_rets["BL"]) / 2
)

factors = factors.dropna()
print(f"Constructed factors: {len(factors)} months")
print(f"\nSummary statistics (monthly):")
display(factors.describe().round(4))

## 4. Download official FF factors and compare

In [ ]:
import pandas_datareader.data as web

# Download official Fama-French 5-factor data
ff_official = web.DataReader(
    "F-F_Research_Data_5_Factors_2x3", "famafrench",
    start="1970", end="2026",
)[0] / 100  # convert from percent to decimal
ff_official.index = ff_official.index.to_timestamp()

print(f"Official FF data: {len(ff_official)} months")
ff_official.head()

In [ ]:
# ── Merge our factors with official ───────────────────────────────────
# Our factors are already dated to the month the return was realized
# (after the +1 month shift above). FF official data uses the same
# convention: the July row contains the July return.
#
# We align by normalizing both to month-start timestamps.

our = factors.copy()
our.index = pd.to_datetime(our.index).to_period("M").to_timestamp()

ff_off = ff_official[["SMB", "HML"]].copy()
ff_off.index = pd.to_datetime(ff_off.index).to_period("M").to_timestamp()

comparison = (
    our.rename(columns={"SMB": "Our SMB", "HML": "Our HML"})
    .join(ff_off.rename(columns={"SMB": "FF SMB", "HML": "FF HML"}), how="inner")
    .dropna()
)

print(f"Overlapping months: {len(comparison)}")
print(f"Period: {comparison.index.min().date()} to {comparison.index.max().date()}")

## 5. Statistical comparison

In [ ]:
from scipy import stats
from statsmodels.tsa.stattools import coint
import statsmodels.api as sm


def compare_factor(our_col, ff_col, name):
    """Run full comparison: correlation, regression, cointegration."""
    y = comparison[our_col].values
    x = comparison[ff_col].values

    # 1. Pearson correlation
    corr, corr_p = stats.pearsonr(y, x)

    # 2. OLS regression: Our = alpha + beta * Official
    X_ols = sm.add_constant(x)
    model = sm.OLS(y, X_ols).fit()
    alpha = model.params[0]
    beta = model.params[1]
    r2 = model.rsquared

    # 3. Cointegration test
    coint_stat, coint_p, crit_vals = coint(y, x)

    print(f"\n{'=' * 60}")
    print(f"  {name} FACTOR COMPARISON: Our vs Official FF")
    print(f"{'=' * 60}")
    print(f"  Pearson Correlation:  {corr:.4f}  (p={corr_p:.2e})")
    print(f"  Regression Alpha:     {alpha:.6f}")
    print(f"  Regression Beta:      {beta:.4f}")
    print(f"  R-squared:            {r2:.4f}")
    print(f"  Cointegration stat:   {coint_stat:.4f}  (p={coint_p:.4f})")
    print(f"    Critical: 1%={crit_vals[0]:.2f}, 5%={crit_vals[1]:.2f}, 10%={crit_vals[2]:.2f}")
    print(f"\n  Interpretation:")
    if corr > 0.9:
        print(f"    Correlation: Excellent (>{0.9})")
    elif corr > 0.8:
        print(f"    Correlation: Good (>{0.8})")
    else:
        print(f"    Correlation: Moderate ({corr:.2f})")
    if abs(beta - 1.0) < 0.1:
        print(f"    Beta: Near 1.0 — good scale match")
    else:
        print(f"    Beta: {beta:.2f} — some scale difference")
    if coint_p < 0.05:
        print(f"    Cointegration: Yes (p<0.05) — long-run relationship")
    else:
        print(f"    Cointegration: No (p={coint_p:.2f})")

    return {"corr": corr, "beta": beta, "r2": r2, "alpha": alpha,
            "coint_p": coint_p}


smb_stats = compare_factor("Our SMB", "FF SMB", "SMB")
hml_stats = compare_factor("Our HML", "FF HML", "HML")

## 6. Visual comparison

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for i, (factor, our_col, ff_col) in enumerate([
    ("SMB", "Our SMB", "FF SMB"),
    ("HML", "Our HML", "FF HML"),
]):
    # Cumulative returns
    ax = axes[i, 0]
    cum_our = (1 + comparison[our_col]).cumprod()
    cum_ff = (1 + comparison[ff_col]).cumprod()
    ax.plot(comparison.index, cum_our, label=f"Our {factor}", linewidth=2)
    ax.plot(comparison.index, cum_ff, label=f"Official {factor}",
            linestyle="--", linewidth=2, alpha=0.8)
    ax.set_title(f"Cumulative Returns: {factor}")
    ax.set_ylabel("Growth of $1")
    ax.legend()
    ax.grid(True, alpha=0.3)

    # Scatter plot
    ax = axes[i, 1]
    ax.scatter(comparison[ff_col], comparison[our_col], alpha=0.3, s=10)
    # 45-degree line
    lims = [min(comparison[ff_col].min(), comparison[our_col].min()),
            max(comparison[ff_col].max(), comparison[our_col].max())]
    ax.plot(lims, lims, "r--", alpha=0.5, label="45° line")
    ax.set_xlabel(f"Official {factor}")
    ax.set_ylabel(f"Our {factor}")
    ax.set_title(f"{factor}: Our vs Official (scatter)")
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ── Rolling correlation ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for i, (factor, our_col, ff_col) in enumerate([
    ("SMB", "Our SMB", "FF SMB"),
    ("HML", "Our HML", "FF HML"),
]):
    rolling_corr = comparison[our_col].rolling(60).corr(comparison[ff_col])
    ax = axes[i]
    ax.plot(comparison.index, rolling_corr, linewidth=1.5)
    ax.axhline(0.9, color="green", linestyle="--", alpha=0.5, label="0.9")
    ax.axhline(0.8, color="orange", linestyle="--", alpha=0.5, label="0.8")
    ax.set_title(f"{factor}: 60-month Rolling Correlation")
    ax.set_ylabel("Correlation")
    ax.set_ylim(0.4, 1.05)
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Performance summary

In [ ]:
# ── Side-by-side performance stats ────────────────────────────────────
from quant_trading.evaluation import perf_stats_annualized

summary = pd.DataFrame({
    col: perf_stats_annualized(comparison[col])
    for col in ["Our SMB", "FF SMB", "Our HML", "FF HML"]
}).T

display(summary.style.format({
    "Annualized Return": "{:.2%}",
    "Annualized Vol": "{:.2%}",
    "Sharpe": "{:.3f}",
    "Max Drawdown": "{:.2%}",
}))

## 8. Tracking error analysis

In [ ]:
# ── Tracking error: difference between our factor and official ────────
for factor in ["SMB", "HML"]:
    diff = comparison[f"Our {factor}"] - comparison[f"FF {factor}"]
    te_ann = diff.std() * np.sqrt(12)
    mean_diff = diff.mean() * 12
    print(f"{factor}:")
    print(f"  Annualized tracking error:  {te_ann:.4f} ({te_ann*100:.2f}%)")
    print(f"  Annualized mean difference: {mean_diff:.4f} ({mean_diff*100:.2f}%)")
    print()

## Notes on expected differences

Our JKP-based replication will **not** match the official FF factors exactly.
Known sources of difference:

1. **Book equity construction**: FF uses a specific hierarchy
   (`seq + txditc - ps`, with fallbacks). JKP's `be_me` uses a similar
   but not identical definition.
2. **Market equity aggregation**: FF aggregates ME across all share classes
   (`permco` level) using the largest `permno`. JKP uses its own ME definition.
3. **Delisting returns**: FF incorporates CRSP delisting returns via
   `retadj = (1+ret)*(1+dlret)-1`. JKP's `ret_exc_lead1m` handles
   delistings differently.
4. **Value-weighting**: FF uses a lagged ME scheme that updates weights
   within the holding year using cumulative ex-dividend returns. We use
   beginning-of-month `market_equity` from JKP.
5. **Universe timing**: FF sorts in June using December fiscal-year-end data
   with a 6-month lag. JKP's `be_me` may have different data availability timing.

A correlation of **0.85+** for HML and **0.90+** for SMB would indicate a
strong replication. Your CRSP/Compustat code achieved 0.82 correlation for
HML vs JKP — so a similar range is expected here.